# Notebook 7: RAG with an Open-Weight Model

**Who this is for:** Digital Humanities students with no prior AI or programming background.

**What you will learn**
1. How to run the full RAG pipeline with a free model on your own Colab GPU: no API key, no cost per question
2. How the answers differ **with** and **without** retrieved sources
3. How a small model compares with the API models of Notebook 5

**Before you start:** this notebook combines Notebook 5 (RAG with a vector database) and Notebook 6 (running an open-weight model). Turn on the free GPU: *Runtime > Change runtime type > T4 GPU > Save*. Lines that start with `#` are comments for humans.

## 0. Setup

In [ ]:
# Check that we have a GPU, and install the tools we need.
import torch
print("GPU found:" if torch.cuda.is_available() else "No GPU. Turn it on in Runtime > Change runtime type.",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# "!" runs a system command. This takes about a minute.
!pip install -q -U transformers accelerate

In [ ]:
# Also install the vector database and the embedding library (same as Notebook 5).
!pip install -q chromadb sentence-transformers pandas matplotlib

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Qwen is open: no account and no token are needed (see Notebook 6 for the details).
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# The tokenizer cuts text into small pieces ("tokens"); the model reads and writes them.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# The first run downloads about 6 GB (a few minutes).
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
print("Model loaded:", MODEL_ID)

In [ ]:
def ask(prompt, system=None, max_tokens=300):
    """Send a question to the local model and return its answer.
    The model always picks its most likely words, so the same question gives the same answer."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    # Turn the conversation into tokens and move them to the GPU.
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,
                                temperature=None, top_p=None, top_k=None,
                                pad_token_id=tokenizer.eos_token_id)

    # Keep only the new words (not our question) and turn them back into text.
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask("Reply with one word: ready", max_tokens=10))

## 1. Build the database

The same invented *Cedar Valley Reading Society* archive as in Notebooks 4 and 5. **Everything in it is invented.**

In [ ]:
#@title Load the invented Cedar Valley archive (double-click to show the code) { display-mode: "form" }
ARCHIVE = [
    {"id": "r01", "title": "Founding",
     "en": "The Cedar Valley Reading Society was founded in 1921 by the teacher Salim Haddad in a rented room above a bakery.",
     "ar": "تأسست جمعية وادي الأرز للقراءة عام 1921 على يد المعلم سليم حداد في غرفة مستأجرة فوق مخبز."},
    {"id": "r02", "title": "Library size",
     "en": "In 1930 the library of the society held 1,200 volumes, mostly Arabic poetry, history books and translated novels.",
     "ar": "في عام 1930 ضمّت مكتبة الجمعية 1200 مجلد، معظمها دواوين شعر عربي وكتب تاريخ وروايات مترجمة."},
    {"id": "r03", "title": "Members in 1928",
     "en": "The lending register for 1928 lists 340 members. Each member could borrow two books for fourteen days.",
     "ar": "يسجّل دفتر الإعارة لعام 1928 أسماء 340 عضوًا. وكان يحق لكل عضو استعارة كتابين لمدة أربعة عشر يومًا."},
    {"id": "r04", "title": "Late fines",
     "en": "A book returned late cost the borrower a fine of one piaster per day. The money was used to repair damaged books.",
     "ar": "كانت غرامة تأخير إعادة الكتاب قرشًا واحدًا عن كل يوم، وكان المال يُستخدم لإصلاح الكتب التالفة."},
    {"id": "r05", "title": "New building",
     "en": "In 1936 the society moved to a new building near the old market. The move was paid for by donations from local merchants.",
     "ar": "في عام 1936 انتقلت الجمعية إلى مبنى جديد قرب السوق القديم، ودُفعت تكاليف الانتقال من تبرعات تجار المدينة."},
    {"id": "r06", "title": "Monthly bulletin",
     "en": "From 1925 to 1939 the society published a monthly bulletin. Only 61 issues survive today, and they are kept in the municipal archive.",
     "ar": "من عام 1925 إلى عام 1939 أصدرت الجمعية نشرة شهرية. لم يبق منها اليوم سوى 61 عددًا، وهي محفوظة في أرشيف البلدية."},
    {"id": "r07", "title": "Women's reading circle",
     "en": "In 1931 the society started a weekly evening reading circle for women, led by the teacher Nadia Khoury.",
     "ar": "في عام 1931 بدأت الجمعية حلقة قراءة مسائية أسبوعية للنساء، أدارتها المعلمة نادية خوري."},
    {"id": "r08", "title": "Poetry lecture",
     "en": "In 1927 the visiting poet Khalil Saba gave a lecture on modern Arabic poetry, and about 200 people attended.",
     "ar": "في عام 1927 ألقى الشاعر الزائر خليل سابا محاضرة عن الشعر العربي الحديث، وحضرها نحو 200 شخص."},
    {"id": "r09", "title": "Merchant's donation",
     "en": "In 1933 the merchant Yusuf Kassab donated 300 books from his private library, including a rare copy of a seventeenth-century geography manuscript.",
     "ar": "في عام 1933 تبرّع التاجر يوسف كسّاب بـ 300 كتاب من مكتبته الخاصة، من بينها نسخة نادرة من مخطوط جغرافي يعود إلى القرن السابع عشر."},
    {"id": "r10", "title": "Closure",
     "en": "The society closed in 1951 when its funds ran out. Its books were kept in a school storeroom, and about 900 volumes were catalogued again in 1990.",
     "ar": "أُغلقت الجمعية عام 1951 بعد نفاد أموالها. حُفظت كتبها في مخزن مدرسة، وأُعيد فهرسة نحو 900 مجلد منها عام 1990."},
    {"id": "r11", "title": "Digitisation",
     "en": "In 2019 a university team digitised the surviving bulletins and the 1928 lending register. The scans can be read online for free.",
     "ar": "في عام 2019 رقمن فريق جامعي النشرات الباقية ودفتر الإعارة لعام 1928، ويمكن قراءة النسخ الممسوحة مجانًا على الإنترنت."},
    {"id": "r12", "title": "Reading habits",
     "en": "The 1928 register shows that poetry was the most borrowed category, followed by history and then translated novels.",
     "ar": "يُظهر سجل عام 1928 أن الشعر كان أكثر الأصناف استعارة، يليه التاريخ ثم الروايات المترجمة."},
]

In [ ]:
import pandas as pd
from IPython.display import display
import chromadb
from sentence_transformers import SentenceTransformer

# The multilingual embedding model (the first run downloads about 450 MB).
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Store every record twice (English and Arabic), each with its labels.
ids, texts, metas = [], [], []
for r in ARCHIVE:
    for lang in ["en", "ar"]:
        ids.append(f"{r['id']}_{lang}")
        texts.append(r[lang])
        metas.append({"record": r["id"], "title": r["title"], "language": lang})

db = chromadb.EphemeralClient()
try:
    db.delete_collection("archive")
except Exception:
    pass
collection = db.create_collection(name="archive", metadata={"hnsw:space": "cosine"})
collection.add(ids=ids, embeddings=embedder.encode(texts).tolist(), documents=texts, metadatas=metas)
print("Stored", collection.count(), "texts")

## 2. The RAG functions

The same three steps as in Notebook 5. Only `ask()` is different: it now calls the model on your own GPU.

In [ ]:
def retrieve(question, k=3, language=None):
    """Step 1 - RETRIEVE: the k stored texts closest in meaning to the question."""
    options = dict(query_embeddings=[embedder.encode(question).tolist()], n_results=k)
    if language:
        options["where"] = {"language": language}     # "en" or "ar"
    found = collection.query(**options)
    return [{"score": round(1 - found["distances"][0][i], 2),
             "title": found["metadatas"][0][i]["title"],
             "text": found["documents"][0][i]}
            for i in range(len(found["ids"][0]))]

def build_prompt(question, passages):
    """Step 2 - AUGMENT: put the passages in the prompt as numbered sources."""
    sources = "".join(f"[{n}] {p['text']}\n" for n, p in enumerate(passages, start=1))
    return (
        "Answer the question using ONLY the sources below.\n"
        "Cite the numbers of the sources you used, for example [1].\n"
        "If the sources do not contain the answer, say exactly: The sources do not say.\n"
        "Answer in the same language as the question.\n\n"
        f"Sources:\n{sources}\n"
        f"Question: {question}"
    )

def rag(question, k=3, language=None):
    """Run the three steps. Returns the answer and the passages that were used."""
    passages = retrieve(question, k=k, language=language)
    answer = ask(build_prompt(question, passages))
    return answer, passages

def show(question, k=3, language=None):
    """Run RAG and print the answer together with its sources."""
    answer, passages = rag(question, k=k, language=language)
    print("Question:", question)
    print("Answer  :", answer)
    print("\nSources used:")
    display(pd.DataFrame(passages)[["score", "title", "text"]])

In [ ]:
show("Who founded the society, and where did it first meet?", language="en")

In [ ]:
# Arabic question, Arabic texts. ("What was the fine for returning a book late?")
show("كم كانت غرامة تأخير إعادة الكتاب؟", language="ar")

## 3. With and without sources

The table puts the same questions to the model **twice**: once with no sources, once with RAG. Small models invent answers more often than large ones, so this comparison is important.

In [ ]:
questions = [
    "How many members did the Cedar Valley Reading Society have in 1928?",
    "What was the fine for returning a book late?",
    "Who led the women's reading circle?",
]

rows = []
for q in questions:
    answer_with_rag, _ = rag(q, language="en")
    rows.append({"Question": q,
                 "Without sources": ask(q, max_tokens=80),      # the model alone
                 "With RAG": answer_with_rag})                  # the model with retrieved sources

pd.set_option("display.max_colwidth", None)                     # do not cut long answers
pd.DataFrame(rows)

**What to look for:** without sources the model usually invents numbers and names, or says it does not know. With RAG the answers match the records. Check each answer against the archive in section 1.

## 4. A question the archive cannot answer

In [ ]:
show("What was the yearly budget of the society?", language="en")

A small model does not always obey the rule "say: The sources do not say". It may still guess. This is the main reason to **read the sources** and to **test the model on questions you know the answer to** before trusting it.

## Open-weight model or API?

| | Open-weight model (this notebook) | API model (Notebook 5) |
|---|---|---|
| Cost | free | pay per use |
| Your texts | stay in your Colab session | are sent to the provider |
| Following instructions | weaker, especially in Arabic | stronger |
| Setup | GPU and a large download | an API key |

For sensitive material, an open-weight model on your own computer or university server is the safer choice. For the best quality, use an API model.

## Key takeaways

- The RAG pipeline is the same with any model: **retrieve, augment, generate**. Only `ask()` changes.
- A small model gives **useful but less reliable** answers. Always check the sources.
- Compare with and without sources on **your own questions** before you rely on results.

## Exercises

1. Ask the Arabic version of the three questions in section 3 (use `language="ar"`). Is the answer quality the same?
2. Add a record of your own to `ARCHIVE` (copy the format) and ask about it.
3. Ask a question that needs two records, with `k=1` and `k=3`.

**Next:** Notebook 8 uses a language model for five common research tasks: translation, summarisation, sentiment, named entities and relationships.